# `compare_engines` — ตรวจว่า block matching กับ optical flow ให้ผลตรงกันหรือไม่ระบบ **ทันฝน** มี motion engine สองตัวที่คืนผลรูปแบบเดียวกัน| engine | motion | extrapolation ||---|---|---|| `light` | masked block matching + parabolic sub-pixel · เวกเตอร์เดียวทั้งภาพ | semi-Lagrangian เขียนเอง || `pysteps` | `dense_lucaskanade` · motion field รายพิกเซล | `pysteps.extrapolation.semilagrangian` |**คำถามของ notebook นี้** — ตอบสามข้อ ตามลำดับความสำคัญ1. **skill** สองตัวพยากรณ์ได้ดีเท่ากันไหม (CSI / POD / FAR เทียบ observation จริง) ← ข้อชี้ขาด2. **agreement** field ที่พยากรณ์ออกมาต่างกันแค่ไหน (MAE dBZ, correlation)3. **ขอบเขต** ความต่างโผล่ตอนไหน — stratify ตามความซับซ้อนของสนามลมข้อ 1 สำคัญกว่าข้อ 2 เพราะสอง engine อาจให้ field ต่างกันแต่ skill เท่ากัน(ต่างกันในที่ที่ไม่มีฝน) หรือ field ใกล้กันแต่ skill ต่าง (ต่างกันตรงขอบก้อนฝนพอดี)> **บริบท** — ก่อน 14 ก.ย. 2026 `requirements.txt` ไม่มี pysteps ระบบจึงตกไปใช้ `light`> ทุกครั้งตั้งแต่เริ่มเดิน notebook นี้คือการตรวจที่ docstring ของ `nowcast.py` เขียนไว้ว่า> *"ก่อนเชื่อ engine เบา ต้องรัน `compare_engines()` บน Colab ที่ลง pysteps ได้"*---### เวลาที่ใช้โดยประมาณวัดจริงบนเครื่องทดสอบ: build stack **~2 วินาที/เฟรม** · เปรียบเทียบ **~1.2 วินาที/origin**(หัวข้อ 6, 8, 9 วนข้อมูลชุดเดิมคนละรอบ รวมเป็นสามรอบ — ตัวเลขข้างล่างรวมทั้งสามแล้ว)| ขั้น | QUICK=True | QUICK=False ||---|---|---|| ติดตั้ง + clone | ~4 นาที | ~4 นาที || build stack (ทำครั้งเดียว แล้ว cache) | ~12 นาที | ~35 นาที || เปรียบเทียบ (§6+§8+§9) | ~7 นาที | ~16 นาที |> `QUICK=True` เลือก**ช่วงต่อเนื่องที่ยาวที่สุด**ช่วงเดียว ซึ่งในคลังนี้คือ 362 เฟรม —> ยาวพอจะให้ตัวเลขที่มีความหมาย แต่ยังเป็นสภาพอากาศชุดเดียว อย่าเอาไปอ้างในเปเปอร์> ถ้าอยากทดสอบว่าท่อทำงานเร็ว ๆ ให้ตั้ง `STRIDE = 4` ด้วย จะเหลือราว 2 นาที

## 1 · ติดตั้ง

In [ ]:
# pysteps ไม่มี wheel บน PyPI ต้อง build จาก sdist — ใช้เวลาประมาณ 1 นาที# opencv จำเป็นสำหรับ dense_lucaskanade (morphologyEx) ถ้าขาดจะ error MissingOptionalDependency!pip install -q pysteps opencv-python-headless pytesseract!apt-get -qq install -y tesseract-ocr > /dev/nullimport importlib.metadata as _mdfrom pysteps.motion.lucaskanade import dense_lucaskanadefrom pysteps.extrapolation import semilagrangianprint("pysteps", _md.version("pysteps"), "· opencv", _md.version("opencv-python-headless"))

## 2 · ดึงโค้ดและคลังข้อมูล

In [ ]:
import os, shutilfrom pathlib import PathREPO = "https://github.com/jamorn12/tmd-radar-archive.git"ROOT = Path("/content/tmd-radar-archive")if not ROOT.exists():    !git clone --depth 1 -q {REPO} {ROOT}else:    print("มีอยู่แล้ว — ข้าม clone (ถ้าอยากดึงใหม่ให้ลบโฟลเดอร์ก่อน)")os.chdir(ROOT)import sys; sys.path.insert(0, str(ROOT))n_raw = len(list((ROOT / "data/raw/PHS").rglob("*.jpg")))print(f"เฟรมดิบในคลัง: {n_raw}")

## 3 · ตั้งค่า`QUICK = True` ใช้เฉพาะช่วงต่อเนื่องที่ยาวที่สุดช่วงเดียว — พอสำหรับดูว่าท่อทำงานไหม`QUICK = False` ใช้ทุกช่วงในคลัง — ตัวเลขนี้เท่านั้นที่เอาไปอ้างในเปเปอร์ได้

In [ ]:
QUICK     = True        # <-- เปลี่ยนเป็น False เมื่อจะเอาตัวเลขจริงไปใส่เปเปอร์STATION   = "PHS"MIN_RUN   = 20          # ช่วงต่อเนื่องที่สั้นกว่านี้ข้ามไปSTRIDE    = 1           # 1 = ใช้ทุก origin · 2 = เว้นอันเว้นอัน (เร็วขึ้นเท่าตัว)STACK_DIR = Path("/content/stacks")import numpy as np, warningsfrom radar_archive import nowcast as N, build_stack, gridfrom radar_archive.config import get_station, CONFIG_PATH# pysteps เตือน "Singular matrix during outlier detection" เป็นครั้งคราว# เกิดตอน LK หา feature ได้น้อยจนเมทริกซ์ degenerate — มันจัดการเองได้ ไม่กระทบผลwarnings.filterwarnings("ignore", message=".*Singular matrix.*")st    = get_station(STATION, str(CONFIG_PATH))LEADS = N.LEADS_MINN_IN  = N.N_INPUTTHR   = 11.98           # dBZ = 0.1 มม./ชม. ภายใต้ Rosenfeld tropical — ใช้ threshold เดียวกันทั้งสองฝั่งprint(f"station {st.code} · leads {LEADS} นาที · ใช้ {N_IN} เฟรมย้อนหลัง · threshold {THR} dBZ")

## 4 · สร้าง stack (ทำครั้งเดียว แล้ว cache)แปลง JPEG → dBZ grid 241×241 ที่ 2 กม. ใช้เวลาราว **2 วินาทีต่อเฟรม**ผลเก็บเป็น `.npz` ต่อหนึ่งช่วงต่อเนื่อง รันซ้ำจะข้ามไฟล์ที่มีแล้ว

In [ ]:
import timeSTACK_DIR.mkdir(parents=True, exist_ok=True)frames = build_stack.find_frames(Path("data"), st.code)runs   = [r for r in build_stack.split_runs(frames) if len(r) >= MIN_RUN]runs.sort(key=len, reverse=True)if QUICK:    runs = runs[:1]print(f"ช่วงที่จะใช้ {len(runs)} ช่วง · รวม {sum(len(r) for r in runs)} เฟรม")for r in runs:    print(f"  {r[0][0]:%d %b %H:%M}-{r[-1][0]:%H:%M}Z  {len(r):3d} เฟรม")t0 = time.time()paths = []for r in runs:    p = STACK_DIR / f"{st.code}_{build_stack.run_name(r)}_mean.npz"    if p.exists():        print(f"[cache] {p.name}")    else:        stack, times, meta, _ = build_stack.build_run(r, st, Path("data"), verbose=False)        grid.save_stack(p, stack, times, meta)        print(f"[build] {p.name}  shape {stack.shape}")    paths.append(p)print(f"\nรวม {time.time()-t0:.0f} วินาที")

## 5 · ฟังก์ชันหลัก**contingency table** — นับ hit / false alarm / miss ที่ threshold เดียวกันทั้งของจริงและพยากรณ์สะสมข้ามทุก origin แล้วค่อยหารทีเดียว (*pooled*) ไม่ใช่เฉลี่ย CSI รายอันแล้วเฉลี่ยอีกทีเพราะ origin ที่ฝนน้อยจะมี CSI แกว่งสุดขั้วและถ่วงค่าเฉลี่ยผิดสัดส่วน

In [ ]:
def contingency(fc, ob, thr=THR):    ok = np.isfinite(fc) & np.isfinite(ob)    f, o = fc[ok] >= thr, ob[ok] >= thr    return np.array([(f & o).sum(), (f & ~o).sum(), (~f & o).sum()], dtype=np.int64)def scores(h, fa, m):    csi = h / (h + fa + m) if (h + fa + m) else np.nan    pod = h / (h + m)      if (h + m)      else np.nan    far = fa / (h + fa)    if (h + fa)     else np.nan    return csi, pod, fardef flow_complexity(V):    '''สนามลมนี้ uniform แค่ไหน — ตัวชี้ว่า light น่าจะพอหรือไม่พอ    light คืนเวกเตอร์เดียวทั้งภาพ ถ้าของจริง uniform อยู่แล้วมันก็ไม่เสียอะไร    แต่ถ้ามี rotation / deformation / ฝนหลายก้อนคนละทิศ เวกเตอร์เดียวย่อมแทนไม่ได้    '''    mag = np.hypot(V[0], V[1]); sel = mag > 0.1    if sel.sum() < 50:        return None    ang = np.arctan2(V[1][sel], V[0][sel])    spd = mag[sel]    return dict(dir_consistency=float(np.hypot(np.cos(ang).mean(), np.sin(ang).mean())),                speed_rel_spread=float(spd.std() / max(spd.mean(), 1e-6)))

### วนทุก originแต่ละ origin `i` ใช้ `stack[i-3..i]` เป็น input แล้วเทียบผลกับ `stack[i+1..i+8]` ที่สังเกตได้จริง`persist` คือ baseline ที่สมมติว่าฝนไม่ขยับเลย — ถ้า engine ชนะ persistence ไม่ได้ ก็ไม่มีประโยชน์ถ้า `estimate_motion` ตกกลับไปใช้อีก engine (เช่นขอ pysteps แต่ import ไม่ได้)จะนับเป็น fail ไม่เอามารวม — ไม่งั้นตัวเลขสอง engine จะปนกันโดยไม่รู้ตัว

In [ ]:
def compare_run(stack, kmperpixel=2.0, timestep=15.0, stride=STRIDE):    steps = [int(round(l / timestep)) for l in LEADS]    engines = ("light", "pysteps")    acc  = {e: {l: np.zeros(3, np.int64) for l in LEADS} for e in engines + ("persist",)}    diff = {l: [] for l in LEADS}          # สะสม |light - pysteps| ทีละ origin    recs, fails = [], {e: 0 for e in engines}    for i in range(N_IN - 1, len(stack) - max(steps), stride):        inp, _ = N.despeckle_stack(stack[i - N_IN + 1: i + 1])        truth  = {l: stack[i + s] for l, s in zip(LEADS, steps)}        for l in LEADS:            acc["persist"][l] += contingency(stack[i], truth[l])        rec, F = {"i": i}, {}        for e in engines:            try:                V, info = N.estimate_motion(inp, e, kmperpixel, timestep)                if info["engine"] != e:                    raise RuntimeError(f"ตกไปใช้ {info['engine']}")                fc, _ = N.run_extrapolation(inp[-1], V, e, LEADS, timestep)                F[e] = fc                for l, f in zip(LEADS, fc):                    acc[e][l] += contingency(f, truth[l])                s = N.motion_stability(V, info, kmperpixel, timestep)                rec[e] = dict(kmh=s.get("kmh"), bearing=s.get("bearing"))                if e == "pysteps":                    rec["flow"] = flow_complexity(V)            except Exception as ex:                fails[e] += 1                rec[e] = dict(error=f"{type(ex).__name__}: {ex}")        if len(F) == 2:                     # agreement ระหว่างสอง engine            for k, l in enumerate(LEADS):                a, b = F["light"][k], F["pysteps"][k]                ok = np.isfinite(a) & np.isfinite(b)                if ok.sum():                    diff[l].append(float(np.abs(a - b)[ok].mean()))        recs.append(rec)    return acc, diff, recs, fails

## 6 · รัน

In [ ]:
from collections import defaultdictACC  = {e: defaultdict(lambda: np.zeros(3, np.int64)) for e in ("light", "pysteps", "persist")}DIFF = defaultdict(list)RECS, FAILS = [], {"light": 0, "pysteps": 0}t0 = time.time()for p in paths:    dbz, times = grid.load_stack(p)    kpp = float(np.load(p)["kmperpixel"])    acc, diff, recs, fails = compare_run(dbz, kmperpixel=kpp)    for e in acc:        for l in LEADS:            ACC[e][l] += acc[e][l]    for l in LEADS:        DIFF[l] += diff[l]    RECS += recs    for e in fails:        FAILS[e] += fails[e]    print(f"{p.name}  {len(recs):4d} origins  fails {fails}")dt = time.time() - t0print(f"\nรวม {len(RECS)} origins · {dt:.0f} วินาที ({dt/max(len(RECS),1):.2f} s/origin) · fails {FAILS}")

## 7 · ตาราง 1 — forecast skillนี่คือข้อชี้ขาด ถ้า CSI ของสอง engine ต่างกันน้อยกว่าความไม่แน่นอนของตัวมันเองก็อ้างได้ว่าระบบจริงใช้ `light` โดยไม่เสีย skill`skill` = CSI − CSI ของ persistence — ถ้าติดลบแปลว่าแพ้การเดาว่าฝนไม่ขยับ

In [ ]:
import pandas as pdrows = []for l in LEADS:    r = {"lead_min": l}    for e in ("light", "pysteps", "persist"):        csi, pod, far = scores(*ACC[e][l])        r[f"CSI_{e}"] = round(csi, 4)        if e != "persist":            r[f"POD_{e}"] = round(pod, 4)            r[f"FAR_{e}"] = round(far, 4)    r["dCSI"]       = round(r["CSI_pysteps"] - r["CSI_light"], 4)    r["skill_light"]   = round(r["CSI_light"]   - r["CSI_persist"], 4)    r["skill_pysteps"] = round(r["CSI_pysteps"] - r["CSI_persist"], 4)    r["MAE_dbz"]    = round(float(np.mean(DIFF[l])), 3) if DIFF[l] else np.nan    rows.append(r)tab = pd.DataFrame(rows)cols = ["lead_min", "CSI_light", "CSI_pysteps", "dCSI", "CSI_persist",        "skill_light", "skill_pysteps", "POD_light", "POD_pysteps",        "FAR_light", "FAR_pysteps", "MAE_dbz"]tab[cols]

## 8 · ความต่างมีนัยสำคัญไหม`dCSI` ตัวเดียวไม่พอ ต้องรู้ว่ามันใหญ่กว่าความผันผวนระหว่าง origin แค่ไหนใช้ **bootstrap** สุ่ม origin คืนที่ 1,000 รอบ แล้วดูช่วง 95%ถ้าช่วงคร่อม 0 แปลว่าข้อมูลเท่าที่มี**ยังแยกไม่ออก**ว่าตัวไหนดีกว่า

In [ ]:
def bootstrap_dcsi(stack_paths, n_boot=1000, seed=0):    '''สุ่ม origin คืนที่ แล้วคำนวณ dCSI ใหม่ทุกรอบ    ต้องเก็บ contingency รายอันไว้ก่อน จึงรันซ้ำอีกรอบแบบเก็บละเอียด    '''    per = {e: {l: [] for l in LEADS} for e in ("light", "pysteps")}    for p in stack_paths:        dbz, _ = grid.load_stack(p)        kpp = float(np.load(p)["kmperpixel"])        steps = [int(round(l / 15.0)) for l in LEADS]        for i in range(N_IN - 1, len(dbz) - max(steps), STRIDE):            inp, _ = N.despeckle_stack(dbz[i - N_IN + 1: i + 1])            truth = {l: dbz[i + s] for l, s in zip(LEADS, steps)}            for e in ("light", "pysteps"):                try:                    V, info = N.estimate_motion(inp, e, kpp, 15.0)                    if info["engine"] != e:                        raise RuntimeError                    fc, _ = N.run_extrapolation(inp[-1], V, e, LEADS, 15.0)                    for l, f in zip(LEADS, fc):                        per[e][l].append(contingency(f, truth[l]))                except Exception:                    for l in LEADS:                        per[e][l].append(np.zeros(3, np.int64))    rng = np.random.default_rng(seed)    out = []    n = len(per["light"][LEADS[0]])    for l in LEADS:        A = np.array(per["light"][l]); B = np.array(per["pysteps"][l])        d = []        for _ in range(n_boot):            idx = rng.integers(0, n, n)            ca, cb = A[idx].sum(0), B[idx].sum(0)            d.append(scores(*cb)[0] - scores(*ca)[0])        lo, hi = np.percentile(d, [2.5, 97.5])        out.append(dict(lead_min=l, dCSI=round(float(np.mean(d)), 4),                        ci_lo=round(float(lo), 4), ci_hi=round(float(hi), 4),                        แยกออก="ใช่" if lo * hi > 0 else "ไม่ (ช่วงคร่อม 0)"))    return pd.DataFrame(out)boot = bootstrap_dcsi(paths)boot

## 9 · ความต่างโผล่ตอนไหน`light` คืน **เวกเตอร์เดียวทั้งภาพ** ถ้าของจริง uniform อยู่แล้วมันก็ไม่เสียอะไรแต่ถ้ามี rotation, deformation หรือฝนหลายก้อนวิ่งคนละทิศ เวกเตอร์เดียวย่อมแทนไม่ได้แบ่ง origin เป็นสองกลุ่มตาม `dir_consistency` ที่ pysteps วัดได้ (1 = ทิศเดียวกันหมด)ถ้า `light` เสีย skill เฉพาะกลุ่ม non-uniform นั่นคือขอบเขตการใช้งานที่ต้องเขียนในเปเปอร์

In [ ]:
dc = np.array([r["flow"]["dir_consistency"] for r in RECS               if r.get("flow") and "error" not in r.get("pysteps", {})])print(f"dir_consistency: median {np.median(dc):.3f} · p10 {np.percentile(dc,10):.3f} "      f"· p90 {np.percentile(dc,90):.3f} · n={len(dc)}")CUT = float(np.median(dc))print(f"เส้นแบ่ง (median) = {CUT:.3f}")print(f"  uniform     : dir_consistency >= {CUT:.3f}")print(f"  non-uniform : dir_consistency <  {CUT:.3f}")

In [ ]:
def skill_by_group(stack_paths, cut):    acc = {g: {e: {l: np.zeros(3, np.int64) for l in LEADS}               for e in ("light", "pysteps")} for g in ("uniform", "non-uniform")}    cnt = {"uniform": 0, "non-uniform": 0}    for p in stack_paths:        dbz, _ = grid.load_stack(p)        kpp = float(np.load(p)["kmperpixel"])        steps = [int(round(l / 15.0)) for l in LEADS]        for i in range(N_IN - 1, len(dbz) - max(steps), STRIDE):            inp, _ = N.despeckle_stack(dbz[i - N_IN + 1: i + 1])            truth = {l: dbz[i + s] for l, s in zip(LEADS, steps)}            try:                Vp, ip = N.estimate_motion(inp, "pysteps", kpp, 15.0)                fl = flow_complexity(Vp)                if fl is None:                    continue                g = "uniform" if fl["dir_consistency"] >= cut else "non-uniform"                cnt[g] += 1                for e, V, info in (("pysteps", Vp, ip),                                   ("light",) + N.estimate_motion(inp, "light", kpp, 15.0)):                    fc, _ = N.run_extrapolation(inp[-1], V, e, LEADS, 15.0)                    for l, f in zip(LEADS, fc):                        acc[g][e][l] += contingency(f, truth[l])            except Exception:                continue    rows = []    for g in ("uniform", "non-uniform"):        for l in LEADS:            cl = scores(*acc[g]["light"][l])[0]            cp = scores(*acc[g]["pysteps"][l])[0]            rows.append(dict(group=g, n_origins=cnt[g], lead_min=l,                             CSI_light=round(cl, 4), CSI_pysteps=round(cp, 4),                             dCSI=round(cp - cl, 4)))    return pd.DataFrame(rows)strat = skill_by_group(paths, CUT)strat.pivot(index="lead_min", columns="group", values="dCSI")

## 10 · รูปสำหรับเปเปอร์สองแผง — ซ้าย CSI เทียบ lead time · ขวา ความต่าง `dCSI` พร้อมช่วง 95%บันทึกเป็น **SVG** ให้ขยับ/แก้ข้อความได้อิสระใน Illustrator หรือ Inkscape และ PNG 300 dpi สำรอง

In [ ]:
import matplotlib.pyplot as pltfrom matplotlib import rcParamsrcParams.update({"font.family": "DejaVu Sans", "font.size": 9,                 "axes.linewidth": 0.8, "svg.fonttype": "none"})fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.0), constrained_layout=True)L = tab["lead_min"]ax1.plot(L, tab["CSI_light"],   "o-",  c="#1a5f66", lw=1.6, ms=4.5, label="block matching (light)")ax1.plot(L, tab["CSI_pysteps"], "s--", c="#c25a1e", lw=1.6, ms=4.0, label="optical flow (pysteps)")ax1.plot(L, tab["CSI_persist"], "^:",  c="#888888", lw=1.3, ms=4.0, label="persistence")ax1.set_xlabel("Lead time (min)"); ax1.set_ylabel("CSI")ax1.set_xticks(list(LEADS)); ax1.set_ylim(0, None)ax1.grid(alpha=.25, lw=.6); ax1.legend(frameon=False, fontsize=8)ax1.set_title("(a) Forecast skill", loc="left", fontsize=9.5, fontweight="bold")ax2.axhline(0, c="#333", lw=.9)ax2.fill_between(boot["lead_min"], boot["ci_lo"], boot["ci_hi"],                 color="#c25a1e", alpha=.18, lw=0, label="95% CI (bootstrap)")ax2.plot(boot["lead_min"], boot["dCSI"], "s-", c="#c25a1e", lw=1.6, ms=4.0,         label="pysteps − light")ax2.set_xlabel("Lead time (min)"); ax2.set_ylabel("Δ CSI")ax2.set_xticks(list(LEADS)); ax2.grid(alpha=.25, lw=.6)ax2.legend(frameon=False, fontsize=8)ax2.set_title("(b) Engine difference", loc="left", fontsize=9.5, fontweight="bold")fig.savefig("/content/fig_engine_comparison.svg", bbox_inches="tight")fig.savefig("/content/fig_engine_comparison.png", dpi=300, bbox_inches="tight")plt.show()print("บันทึกแล้ว: fig_engine_comparison.svg / .png")

## 11 · บันทึกผลและดาวน์โหลด

In [ ]:
import json as _jsontab.to_csv("/content/engine_comparison_skill.csv", index=False)boot.to_csv("/content/engine_comparison_bootstrap.csv", index=False)strat.to_csv("/content/engine_comparison_stratified.csv", index=False)summary = dict(    station=st.code, quick=QUICK, stride=STRIDE, threshold_dbz=THR,    leads_min=list(LEADS), n_input_frames=N_IN,    n_stacks=len(paths), n_origins=len(RECS), fails=FAILS,    dir_consistency_cut=round(CUT, 4),    pysteps_version=_md.version("pysteps"),    runs=[p.name for p in paths],)Path("/content/engine_comparison_summary.json").write_text(    _json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")print(_json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
from google.colab import filesfor f in ("fig_engine_comparison.svg", "fig_engine_comparison.png",          "engine_comparison_skill.csv", "engine_comparison_bootstrap.csv",          "engine_comparison_stratified.csv", "engine_comparison_summary.json"):    files.download(f"/content/{f}")

## 12 · รัน `compare_engines()` ของ repo เองฟังก์ชันนี้อยู่ใน `radar_archive/nowcast.py` มาตั้งแต่ต้นแต่ไม่เคยถูกรันมันวัด agreement ของ **field** ตรง ๆ (MAE / p95 / max / corr) ต่างจากส่วนบนที่วัด skillรันไว้เพื่อให้โค้ดในระบบถูกใช้จริงอย่างน้อยหนึ่งครั้ง และได้ตัวเลขชุดที่สองมาเทียบ

In [ ]:
dbz, times = grid.load_stack(paths[0])meta = grid.station_meta(st)meta["kmperpixel"] = float(np.load(paths[0])["kmperpixel"])meta["timestep"]   = 15.0res = N.compare_engines(dbz[-N_IN:], meta)print(_json.dumps(res, ensure_ascii=False, indent=2, default=str))

---## วิธีอ่านผล**ถ้า `dCSI` ทุก lead มีช่วง 95% คร่อม 0**ข้อมูลเท่าที่มียังแยกไม่ออกว่า engine ไหนดีกว่า — เขียนในเปเปอร์ได้ว่า*"the two engines were statistically indistinguishable over N origins"*พร้อมระบุ N และช่วง CI อย่าเขียนว่า "เท่ากัน" เพราะไม่ได้พิสูจน์ว่าเท่า แค่แยกไม่ออก**ถ้า `dCSI` เป็นบวกอย่างมีนัยสำคัญ**optical flow ดีกว่าจริง — ระบบควรใช้ `engine="auto"` (ซึ่งตอนนี้เป็นค่าเริ่มต้นแล้ว)และต้องรัน backtest ใหญ่ใหม่ด้วย pysteps เพราะผลเดิมทั้งหมดรันบน `light`**ถ้า `dCSI` ต่างกันเฉพาะกลุ่ม non-uniform** (ตาราง §9)นี่คือผลที่มีค่าที่สุดสำหรับเปเปอร์ — บอกได้ว่า block matching ใช้ได้ภายใต้เงื่อนไขอะไรและพังเมื่อไหร่ ซึ่งตรงกับข้อจำกัดเชิงกายภาพของมัน (เวกเตอร์เดียวแทนทั้งโดเมน)### ข้อควรระวัง- origin ที่อยู่ในช่วงต่อเนื่องเดียวกันไม่เป็นอิสระต่อกัน (ฝนก้อนเดิมโผล่ในหลาย origin)  bootstrap แบบสุ่ม origin จึงให้ CI ที่**แคบเกินจริง** ถ้าจะเข้มงวดควร bootstrap ทีละช่วงต่อเนื่อง- `MAE_dbz` คิดเฉพาะพิกเซลที่ทั้งสอง engine มีค่า — ไม่นับที่ต่างกันเพราะ NaN คนละที่- ผลทั้งหมดผูกกับ threshold 11.98 dBZ เดียว ถ้าเปเปอร์อ้างฝนหนักควรทำซ้ำที่ 35 dBZ ด้วย